# Notebook 2 — Create the Labels

## Objective

Create the target label `late` by comparing the actual delivery date
with the estimated delivery date.

- `late = 1` if the actual delivery date is after the estimated delivery date.
- `late = 0` otherwise.

## Input Artifact

`ml_orders.parquet`

## Output Artifact

`labeled_orders.parquet`

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
input_path = Path("../data/ml_orders.parquet")

In [3]:
df = pd.read_parquet(input_path)

In [4]:
df.shape

(99441, 28)

In [5]:
df.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'item_count',
 'total_item_price',
 'total_freight_value',
 'avg_item_price',
 'payment_count',
 'total_payment_value',
 'max_payment_installments',
 'review_count',
 'avg_review_score',
 'min_review_score',
 'max_review_score',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'unique_product_count',
 'unique_seller_count',
 'unique_category_count',
 'avg_latitude',
 'avg_longitude']

In [6]:
df[
    [
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head(10)

,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-10 21:25:13,2017-10-18
1,2018-08-07 15:27:45,2018-08-13
2,2018-08-17 18:06:29,2018-09-04
3,2017-12-02 00:28:42,2017-12-15
4,2018-02-16 18:17:02,2018-02-26
5,2017-07-26 10:57:55,2017-08-01
6,NaT,2017-05-09
7,2017-05-26 12:55:51,2017-06-07
8,2017-02-02 14:08:10,2017-03-06
9,2017-08-16 17:14:30,2017-08-23


In [7]:
df[
    [
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].isna().sum()

order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [8]:
df.loc[
    df["order_delivered_customer_date"].isna(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [9]:
df_labeled = df.dropna(
    subset=["order_delivered_customer_date"]
).copy()

In [10]:
df_labeled.shape

(96476, 28)

إنشاء late

In [11]:
df_labeled["late"] = (
    df_labeled["order_delivered_customer_date"]
    > df_labeled["order_estimated_delivery_date"]
).astype(int)

نتأكد أن (late) انضاف

In [12]:
df_labeled[[
    "order_id",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "late"
]].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,0
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,0
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,0
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-29 11:18:31,2017-06-07,0


In [18]:
df_labeled["late"].value_counts()

late
0    88649
1     7827
Name: count, dtype: int64

In [14]:
df_labeled["late"].unique()

array([0, 1])

التحقق من عدم وجود Missing في الـLabel

In [15]:
df_labeled["late"].isna().sum()

np.int64(0)

حساب النسب المئوية

In [16]:
label_percentages = (
    df_labeled["late"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

label_percentages

late
0    91.887101
1     8.112899
Name: proportion, dtype: float64

عرض العدد والنسبة معًا

In [19]:
label_distribution = pd.DataFrame({
    "count": df_labeled["late"].value_counts().sort_index(),
    "percentage": df_labeled["late"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
})

label_distribution

,count,percentage
late,,
0,88649,91.887101
1,7827,8.112899


فحص التوزيع بشكل أوضح

In [20]:
print("On-time orders (late = 0):", (df_labeled["late"] == 0).sum())
print("Late orders (late = 1):", (df_labeled["late"] == 1).sum())

print("\nPercentage:")
print(label_percentages)

On-time orders (late = 0): 88649
Late orders (late = 1): 7827

Percentage:
late
0    91.887101
1     8.112899
Name: proportion, dtype: float64


## Class Distribution

After creating the `late` label, the class distribution was examined.

- `late = 0` (on-time): 88,649 orders (91.89%)
- `late = 1` (late): 7,827 orders (8.11%)

The target variable is imbalanced because the majority of orders are on-time,
while late orders represent a much smaller proportion of the dataset.

This class imbalance should be considered later when selecting the baseline,
evaluation metrics, and modeling strategy.

In [21]:
output_path = Path("../data/labeled_orders.parquet")

حفظ البيانات

In [22]:
df_labeled.to_parquet(output_path, index=False)

In [24]:
output_path.stat().st_size

15453504

In [25]:
saved_df = pd.read_parquet(output_path)

In [26]:
saved_df.shape

(96476, 29)

# Summary

Notebook 2 successfully created the target label `late`.

### Label Definition

- `late = 1`: actual delivery date is after the estimated delivery date.
- `late = 0`: actual delivery date is on or before the estimated delivery date.

### Data

- Original orders: 99,441
- Orders with valid actual delivery dates: 96,476
- Orders excluded due to missing actual delivery date: 2,965

### Class Distribution

- On-time (`late = 0`): 88,649 orders (91.89%)
- Late (`late = 1`): 7,827 orders (8.11%)

The target variable is imbalanced, with late orders representing a minority
of the labeled dataset.

### Output Artifact

`labeled_orders.parquet`